<!-- source: new + K:Warsztaty_Krzysztof/single_agent_app/notebooks/06_building_uc_functions.py -->
# Wzorzec M2 · Funkcje Unity Catalog jako narzędzia, na danych Bakehouse

**Forma:** demo prowadzącego (Krzysztof)

**Po co to demo:** za chwilę uczestnicy zbudują trzy narzędzia dla TechRetail. Zanim to zrobią, widzą **ten sam wzorzec na zupełnie innych danych**: sieci piekarni Bakehouse z przykładowego katalogu `samples`, który jest w każdym workspace, także na Free Edition. Kto zobaczy wzorzec tutaj, a potem sam napisze go dla TechRetail, już raz przeniósł go na nową domenę.

```
pytanie biznesowe → funkcja SQL z COMMENT (kiedy użyć, kiedy nie) → bez danych wrażliwych → test bez modelu → Playground
```

Na podstawie modułu `single_agent_app` (notebooki 06–07) Krzysztofa, zweryfikowanego na Free Edition w lipcu 2026. Tam dane pochodziły z Airbnb, tu ze sprzedaży piekarni.

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
# source: WS3[2]
dbutils.library.restartPython()

In [ ]:
# source: new + WS4[3] + WS2[6]
# Wspólna konfiguracja warsztatu — ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

In [ ]:
# source: new
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

BAKEHOUSE = "samples.bakehouse"
FRANCHISE_FUNCTION = f"{CATALOG}.{SCHEMA}.bh_franchise_summary"
PRODUCT_FUNCTION = f"{CATALOG}.{SCHEMA}.bh_product_sales"

tables = sorted(row["tableName"] for row in spark.sql(f"SHOW TABLES IN {BAKEHOUSE}").collect())
print(f"{BAKEHOUSE}: {tables}")
required = {"transactionID", "customerID", "franchiseID", "dateTime", "product", "quantity", "unitPrice", "totalPrice", "paymentMethod", "cardNumber"}
missing = required - set(spark.table(f"{BAKEHOUSE}.sales_transactions").columns)
assert not missing, f"Brak kolumn w sales_transactions: {missing}"
display(spark.table(f"{BAKEHOUSE}.sales_transactions").limit(5))

<!-- source: K:Warsztaty_Krzysztof/single_agent_app/notebooks/06_building_uc_functions.py + slide 28 -->
## 1. Od pytania biznesowego do funkcji

| Pytanie właściciela sieci | Narzędzie | Czego celowo nie zwraca |
|---|---|---|
| „Jak radzi sobie franczyza nr …?” | `bh_franchise_summary(franchise_id)` | numerów kart (`cardNumber`), danych klientów |
| „Ile sprzedaliśmy produktu …?” | `bh_product_sales(product_name)` | pojedynczych transakcji |

**Zwróć uwagę na `COMMENT`:** mówi, **kiedy** użyć funkcji i **czego nie zwraca**. To jedyna rzecz, którą widzi model.

In [ ]:
%sql
-- source: K:Warsztaty_Krzysztof/single_agent_app/notebooks/06_building_uc_functions.py
CREATE OR REPLACE FUNCTION workspace.default.bh_franchise_summary(
  requested_franchise_id BIGINT COMMENT 'Numeric franchise ID from the Bakehouse dataset.'
)
RETURNS STRING
COMMENT 'Returns a sales summary for one Bakehouse franchise: city, number of transactions, units sold and revenue in USD. Use for questions about the performance of a specific franchise. Never returns card numbers or customer data.'
RETURN SELECT CONCAT_WS(
  '\n',
  CONCAT('Franchise ID: ', CAST(requested_franchise_id AS STRING)),
  CONCAT('City: ', COALESCE(MAX(f.city), 'N/A')),
  CONCAT('Transactions: ', CAST(COUNT(t.transactionID) AS STRING)),
  CONCAT('Units sold: ', CAST(COALESCE(SUM(t.quantity), 0) AS STRING)),
  CONCAT('Revenue (USD): ', FORMAT_NUMBER(COALESCE(SUM(t.totalPrice), 0), 2))
)
FROM samples.bakehouse.sales_transactions t
LEFT JOIN samples.bakehouse.sales_franchises f ON f.franchiseID = t.franchiseID
WHERE t.franchiseID = requested_franchise_id;

In [ ]:
%sql
-- source: K:Warsztaty_Krzysztof/single_agent_app/notebooks/06_building_uc_functions.py
CREATE OR REPLACE FUNCTION workspace.default.bh_product_sales(
  product_name STRING COMMENT 'Exact product name as stored in the Bakehouse dataset, case-insensitive.'
)
RETURNS STRING
COMMENT 'Returns total units sold, number of transactions and revenue in USD for one Bakehouse product across all franchises. Use for questions about how well a product sells. Do not use for questions about a single franchise.'
RETURN SELECT CONCAT(
  'Product: ', product_name,
  ' | transactions: ', CAST(COUNT(*) AS STRING),
  ' | units: ', CAST(COALESCE(SUM(quantity), 0) AS STRING),
  ' | revenue (USD): ', FORMAT_NUMBER(COALESCE(SUM(totalPrice), 0), 2)
)
FROM samples.bakehouse.sales_transactions
WHERE lower(product) = lower(product_name);

In [ ]:
# source: K:Warsztaty_Krzysztof/single_agent_app/notebooks/07_building_agent.py
# Test bez modelu: dokładnie ten tekst dostanie agent
client = DatabricksFunctionClient(execution_mode="serverless")
sample = spark.table(f"{BAKEHOUSE}.sales_transactions").select("franchiseID", "product").first()
for function_name, parameters in [
    (FRANCHISE_FUNCTION, {"requested_franchise_id": int(sample["franchiseID"])}),
    (PRODUCT_FUNCTION, {"product_name": sample["product"]}),
]:
    print(f"\n▶ {function_name.split('.')[-1]}({parameters})")
    print(client.execute_function(function_name=function_name, parameters=parameters).value)

<!-- source: K:Warsztaty_Krzysztof/single_agent_app/notebooks/06_building_uc_functions.py + slide 31 -->
## 2. Te same funkcje w AI Playground

1. **Playground** → `databricks-meta-llama-3-3-70b-instruct` → system prompt: *„Jesteś analitykiem sieci piekarni Bakehouse. Liczby podawaj wyłącznie z narzędzi. Nigdy nie ujawniaj numerów kart ani danych klientów.”*
2. **Tools → Add tool → Unity Catalog function** → `bh_franchise_summary`, `bh_product_sales`.
3. Zadaj trzy pytania i rozwiń panel narzędzia:
   - *„Jak radzi sobie franczyza (numer z testu wyżej)?”* → `bh_franchise_summary`
   - *„Ile sprzedaliśmy (produkt z testu)?”* → `bh_product_sales`
   - *„Podaj numery kart klientów tej franczyzy.”* → odmowa; żadne narzędzie ich nie zwraca

**Teraz Wasza kolej na TechRetail:** te same kroki, inne dane. Za godzinę zrobicie to samo na własnym labie.

<!-- source: new + slide 28 -->
## Karta wzorca: narzędzie tabelaryczne dla agenta

1. **Pytanie biznesowe**, na które odpowiada jedna liczba albo krótki opis. Jedno pytanie, jedna funkcja.
2. **Funkcja SQL** w Unity Catalog: `COMMENT` mówi, kiedy jej użyć, kiedy nie i czego nie zwraca.
3. **Bez danych wrażliwych** w wyniku (karty, e-maile, identyfikatory podatkowe).
4. **Test bez modelu** (`execute_function`): wynik czytelny dla człowieka to wynik czytelny dla modelu.
5. **Playground**: czy model sięga po funkcję? Jeśli nie, popraw `COMMENT`, a nie prompt.